In [ ]:
# UCS420 - Cognitive Computing Assignment 10
# Diabetes Care Information Assistant
# Roll number category: Odd
# Run the cells from top to bottom. Designed for Google Colab.

In [ ]:
%pip -q install nltk scikit-learn pandas matplotlib seaborn

In [ ]:
import re
import warnings
warnings.filterwarnings("ignore")

import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.metrics.pairwise import cosine_similarity

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

print("Libraries and NLP resources are ready.")

In [ ]:
# 1. LABELLED CORPUS: 64 queries across 8 intent categories

corpus = {
    "symptoms": [
        "What are the common symptoms of diabetes?",
        "How can I tell if my blood sugar is high?",
        "Why do people with diabetes feel very thirsty?",
        "Can frequent urination be a sign of diabetes?",
        "What are early warning signs of type two diabetes?",
        "I feel tired all the time could it be diabetes?",
        "Does blurred vision happen with high blood glucose?",
        "Can unexplained weight loss be related to diabetes?"
    ],
    "risk_factors": [
        "What increases a person's risk of developing diabetes?",
        "Does family history increase diabetes risk?",
        "Can being overweight lead to type two diabetes?",
        "Does age affect the chance of getting diabetes?",
        "Is high blood pressure linked with diabetes risk?",
        "Can gestational diabetes increase future diabetes risk?",
        "Does physical inactivity increase diabetes risk?",
        "Are some people more likely to develop type two diabetes?"
    ],
    "prevention": [
        "How can I reduce my risk of type two diabetes?",
        "What lifestyle changes help prevent diabetes?",
        "Can regular exercise lower diabetes risk?",
        "How does maintaining a healthy weight help prevent diabetes?",
        "What habits can help me avoid developing diabetes?",
        "Can stopping smoking improve diabetes prevention?",
        "How much physical activity is useful for diabetes prevention?",
        "Are healthy eating habits useful in preventing type two diabetes?"
    ],
    "diet": [
        "What foods are suitable for someone with diabetes?",
        "Should people with diabetes avoid sugary drinks?",
        "What is a balanced meal for diabetes management?",
        "Can a person with diabetes eat fruit?",
        "Why is fibre important in a diabetes-friendly diet?",
        "What are healthier alternatives to refined carbohydrates?",
        "How can I plan meals to manage blood glucose?",
        "Is portion control important for people with diabetes?"
    ],
    "exercise": [
        "What exercises are helpful for people with diabetes?",
        "How often should adults with diabetes be physically active?",
        "Can walking after meals help blood sugar control?",
        "What should I consider before starting a workout with diabetes?",
        "Does strength training help people with diabetes?",
        "Can exercise cause blood glucose to drop?",
        "How can I stay active while managing diabetes?",
        "Why is physical activity recommended for diabetes care?"
    ],
    "monitoring": [
        "How do I monitor my blood glucose at home?",
        "What does an HbA1c test measure?",
        "When should blood sugar be checked?",
        "Why is keeping a glucose log useful?",
        "What is a blood glucose meter used for?",
        "How often should I discuss my HbA1c with my doctor?",
        "What is continuous glucose monitoring?",
        "Why should people with diabetes attend regular checkups?"
    ],
    "medication": [
        "What is metformin commonly prescribed for?",
        "Why might a person with diabetes need insulin?",
        "Should I stop my diabetes medicine if I feel better?",
        "What should I do if I forget a prescribed diabetes dose?",
        "Can diabetes medicines have side effects?",
        "Why is it important to take diabetes medication as prescribed?",
        "Who decides the correct dose of diabetes medicine?",
        "Can I change my insulin dose without medical advice?"
    ],
    "complications": [
        "What health problems can develop from long term diabetes?",
        "How can diabetes affect the kidneys?",
        "Can diabetes damage the nerves in the feet?",
        "Why are regular eye examinations important in diabetes?",
        "Does diabetes increase the risk of heart disease?",
        "How can diabetes affect wound healing?",
        "What is diabetic neuropathy?",
        "Why is foot care important for someone with diabetes?"
    ]
}

records = [
    {"query": query, "intent": intent}
    for intent, queries in corpus.items()
    for query in queries
]
dataset = pd.DataFrame(records)

print("Total labelled queries:", len(dataset))
print("Number of intent categories:", dataset["intent"].nunique())
display(dataset.groupby("intent").size().rename("query_count").to_frame())
display(dataset.sample(10, random_state=42).reset_index(drop=True))

In [ ]:
# 2. HIERARCHICAL TAXONOMY
# Four major categories, each containing at least two subcategories.

taxonomy = {
    "Understanding Diabetes": {
        "Symptoms": ["symptoms", "early_signs", "high_blood_glucose_signs"],
        "Risk and Causes": ["risk_factors", "family_history", "lifestyle_risks"]
    },
    "Lifestyle Management": {
        "Food and Nutrition": ["diet", "meal_planning", "portion_control"],
        "Physical Activity": ["exercise", "aerobic_activity", "strength_training"]
    },
    "Clinical Care": {
        "Testing and Monitoring": ["monitoring", "blood_glucose_testing", "hba1c"],
        "Medicines and Treatment": ["medication", "insulin", "prescribed_treatment"]
    },
    "Health Protection": {
        "Prevention": ["prevention", "healthy_weight", "risk_reduction"],
        "Complications and Screening": ["complications", "eye_health", "kidney_health", "foot_care"]
    }
}

for major_category, subcategories in taxonomy.items():
    print("\n" + major_category)
    for subcategory, topics in subcategories.items():
        print("  ", subcategory, "->", ", ".join(topics))

In [ ]:
# 3. DOMAIN CONCEPTS AND RELATIONSHIPS

domain_concepts = {
    "Diabetes": "A group of conditions involving elevated blood glucose.",
    "Blood Glucose": "The amount of glucose present in the blood.",
    "Insulin": "A hormone that helps glucose enter cells.",
    "Pancreas": "An organ that produces insulin.",
    "HbA1c": "A test that reflects average blood glucose over approximately three months.",
    "Nutrition": "Food choices and eating patterns that influence health and glucose levels.",
    "Physical Activity": "Body movement that can support glucose management and general health.",
    "Medication": "Prescribed treatment used as part of an individual diabetes care plan.",
    "Kidneys": "Organs that filter waste from the blood and can be affected by diabetes.",
    "Eyes": "Organs that can be affected by diabetes-related retinal changes.",
    "Nerves": "Structures that may be damaged by prolonged elevated blood glucose.",
    "Heart": "An organ whose cardiovascular health is important in diabetes care.",
    "Glucose Monitoring": "Checking and recording blood glucose levels."
}

domain_relationships = [
    ("Pancreas", "produces", "Insulin"),
    ("Insulin", "helps regulate", "Blood Glucose"),
    ("Diabetes", "is associated with", "elevated Blood Glucose"),
    ("HbA1c", "provides information about", "average Blood Glucose"),
    ("Nutrition", "can influence", "Blood Glucose"),
    ("Physical Activity", "can support management of", "Blood Glucose"),
    ("Medication", "may be prescribed to manage", "Diabetes"),
    ("Diabetes", "can affect", "Kidneys"),
    ("Diabetes", "can affect", "Eyes"),
    ("Diabetes", "can damage", "Nerves"),
    ("Diabetes", "is associated with increased risk of", "Heart disease"),
    ("Glucose Monitoring", "tracks", "Blood Glucose"),
    ("Insulin", "is a treatment option for some people with", "Diabetes")
]

print("Domain concepts:", len(domain_concepts))
for concept, definition in domain_concepts.items():
    print(f"{concept}: {definition}")

print("\nDomain relationships:", len(domain_relationships))
for subject, relation, obj in domain_relationships:
    print(f"{subject} --{relation}--> {obj}")

In [ ]:
# 4. NLP PREPROCESSING
# Steps: lowercase conversion, tokenization, stopword removal and lemmatization.

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    tokens = word_tokenize(text)
    cleaned_tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens
        if token.isalpha() and token not in stop_words
    ]
    return cleaned_tokens

sample_query = "How can regular exercises help people with Diabetes?"
print("Original:", sample_query)
print("Preprocessed tokens:", preprocess_text(sample_query))

In [ ]:
# 5. TF-IDF FEATURE EXTRACTION AND CLASSIFIER TRAINING

X = dataset["query"]
y = dataset["intent"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

tfidf_vectorizer = TfidfVectorizer(
    tokenizer=preprocess_text,
    token_pattern=None,
    lowercase=False,
    ngram_range=(1, 2),
    min_df=1
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

classifier = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)
classifier.fit(X_train_tfidf, y_train)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])
print("TF-IDF training matrix shape:", X_train_tfidf.shape)
print("Classifier training completed.")

In [ ]:
# 6. MODEL EVALUATION

y_pred = classifier.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

evaluation_results = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Value": [accuracy, precision, recall, f1]
})
display(evaluation_results)

print(classification_report(y_test, y_pred, zero_division=0))

labels = sorted(dataset["intent"].unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)

plt.figure(figsize=(10, 7))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=labels, yticklabels=labels
)
plt.xlabel("Predicted Intent")
plt.ylabel("Actual Intent")
plt.title("Confusion Matrix - Diabetes Care Assistant")
plt.xticks(rotation=35, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# 7. DOMAIN-SPECIFIC KNOWLEDGE BASE
# General educational information only; not a substitute for professional care.

knowledge_base = {
    "symptoms": (
        "Possible diabetes symptoms include increased thirst, frequent urination, "
        "increased hunger, unexplained weight changes, fatigue and blurred vision. "
        "Some people have few or no noticeable symptoms. A healthcare professional "
        "can arrange appropriate testing."
    ),
    "risk_factors": (
        "Type 2 diabetes risk can be associated with family history, increasing age, "
        "overweight or obesity, physical inactivity and a history of gestational diabetes. "
        "Risk varies between individuals; a clinician can assess personal risk."
    ),
    "prevention": (
        "For many people, risk reduction may include regular physical activity, "
        "balanced eating, maintaining a healthy weight where appropriate, avoiding "
        "tobacco and attending preventive health checkups. Prevention is not guaranteed."
    ),
    "diet": (
        "A balanced eating pattern may include non-starchy vegetables, fibre-rich foods, "
        "appropriate portions of whole grains, protein sources and water instead of "
        "sugary drinks. Individual meal plans should consider medicines, preferences "
        "and other health needs; a registered dietitian can help."
    ),
    "exercise": (
        "Regular physical activity can support blood glucose management and overall "
        "health. Walking, cycling, swimming and strength activities may be options. "
        "People using insulin or certain glucose-lowering medicines should ask their "
        "care team how to prevent and respond to exercise-related low blood glucose."
    ),
    "monitoring": (
        "Blood glucose meters measure glucose at a particular time. Continuous glucose "
        "monitoring systems measure glucose trends using a sensor. HbA1c gives an "
        "estimate of average blood glucose over roughly the previous three months. "
        "Testing frequency and targets should be individualized by a clinician."
    ),
    "medication": (
        "Diabetes treatment may include medicines such as metformin or insulin, "
        "depending on the type of diabetes and individual circumstances. Take medicines "
        "as prescribed. Do not start, stop or change a dose based on this assistant; "
        "contact the prescribing clinician or pharmacist about missed doses or side effects."
    ),
    "complications": (
        "Over time, diabetes can increase the risk of problems involving the heart, "
        "kidneys, eyes, nerves and feet. Regular checkups, recommended screening, "
        "glucose management and blood pressure and cholesterol care can help reduce risk."
    )
}

for intent, information in knowledge_base.items():
    print(f"\n{intent.upper()}\n{information}")

In [ ]:
# 8. RULE-BASED REASONING
# Rules include emergency escalation, low-confidence handling and unsupported-query handling.

EMERGENCY_PATTERNS = [
    r"chest pain", r"severe chest pressure", r"cannot breathe", r"can't breathe",
    r"trouble breathing", r"unconscious", r"not responding", r"seizure",
    r"fainted and", r"severe confusion", r"blue lips",
    r"vomiting.*can't keep", r"very difficult to wake"
]

DIABETES_EMERGENCY_PATTERNS = [
    r"blood sugar.{0,20}(below|under)\s*\d+",
    r"glucose.{0,20}(below|under)\s*\d+",
    r"severe low blood sugar", r"severe hypoglycemia",
    r"diabetic ketoacidosis", r"ketones.{0,20}(vomit|breath|high)",
    r"high blood sugar.{0,30}(vomit|confus|breath)"
]

SUPPORTED_TERMS = {
    "diabetes", "diabetic", "blood sugar", "blood glucose", "glucose",
    "insulin", "hba1c", "metformin", "pancreas", "sugar level",
    "diabetes medicine", "diabetes medication", "type 1", "type 2",
    "gestational diabetes", "hypoglycemia", "hyperglycemia"
}

def is_emergency_query(query):
    q = str(query).lower()
    return any(re.search(pattern, q) for pattern in EMERGENCY_PATTERNS + DIABETES_EMERGENCY_PATTERNS)

def is_diabetes_related(query):
    q = str(query).lower()
    return any(term in q for term in SUPPORTED_TERMS)

def classify_query(query):
    vector = tfidf_vectorizer.transform([query])
    probabilities = classifier.predict_proba(vector)[0]
    best_index = int(np.argmax(probabilities))
    predicted_intent = classifier.classes_[best_index]
    confidence = float(probabilities[best_index])

    # Similarity check prevents unrelated text from being forced into an intent.
    similarity_scores = cosine_similarity(vector, X_train_tfidf)
    max_similarity = float(similarity_scores.max()) if similarity_scores.size else 0.0

    return predicted_intent, confidence, max_similarity

def apply_reasoning_rules(query, intent, confidence, similarity):
    # Rule 1: Emergency escalation always takes priority over classification.
    if is_emergency_query(query):
        return (
            "EMERGENCY ESCALATION: This may describe a medical emergency. "
            "Call your local emergency number or go to the nearest emergency department "
            "now. Do not wait for an online assistant. If the person is unconscious, "
            "having a seizure, having severe breathing difficulty or experiencing severe "
            "chest pain, seek emergency help immediately. Follow instructions from "
            "emergency professionals."
        )

    # Rule 2: Unsupported healthcare topics are not answered by this diabetes assistant.
    if not is_diabetes_related(query):
        return (
            "UNSUPPORTED QUERY: I am a Diabetes Care Information Assistant and can "
            "answer general diabetes-related questions only. Please consult a qualified "
            "healthcare professional for this topic or use an appropriate health service."
        )

    # Rule 3: Low confidence or low similarity triggers clarification.
    if confidence < 0.35 or similarity < 0.08:
        return (
            "LOW CONFIDENCE: I could not confidently identify the diabetes-related "
            "topic. Please rephrase your question or provide more details. For personal "
            "medical advice, contact your healthcare professional."
        )

    # Rule 4: Provide only information mapped to a recognized intent.
    if intent not in knowledge_base:
        return (
            "I do not have reliable information for that request in my knowledge base. "
            "Please consult a healthcare professional."
        )

    # Rule 5: Medication questions always receive an additional safety reminder.
    answer = knowledge_base[intent]
    if intent == "medication":
        answer += " Never change a prescribed dose without professional guidance."

    # Rule 6: All informational responses include a non-diagnostic disclaimer.
    answer += " This is general educational information, not a diagnosis or personal treatment plan."
    return answer

def diabetes_assistant(query, verbose=True):
    intent, confidence, similarity = classify_query(query)
    response = apply_reasoning_rules(query, intent, confidence, similarity)

    result = {
        "query": query,
        "predicted_intent": intent,
        "confidence": round(confidence, 3),
        "similarity": round(similarity, 3),
        "response": response
    }

    if verbose:
        print("\nUser:", query)
        print("Predicted intent:", intent)
        print("Confidence:", result["confidence"])
        print("Similarity:", result["similarity"])
        print("Assistant:", response)

    return result

print("Reasoning rules and integrated assistant are ready.")

In [ ]:
# 9. TESTING THE ASSISTANT
# Test one query from every intent category.

intent_test_queries = [
    "What are the common symptoms of diabetes?",
    "Does family history increase diabetes risk?",
    "How can I reduce my risk of type two diabetes?",
    "What foods are suitable for someone with diabetes?",
    "Can walking after meals help blood sugar control?",
    "What does an HbA1c test measure?",
    "Why might a person with diabetes need insulin?",
    "How can diabetes affect the kidneys?"
]

intent_test_results = []
for query in intent_test_queries:
    intent_test_results.append(diabetes_assistant(query))

display(pd.DataFrame(intent_test_results))

In [ ]:
# 10. TEST DIFFERENTLY WORDED QUERY

paraphrased_query = "I keep needing to pee and my mouth feels dry all day. Could high sugar be the reason?"
paraphrase_result = diabetes_assistant(paraphrased_query)
display(pd.DataFrame([paraphrase_result]))

In [ ]:
# 11. TEST UNSUPPORTED HEALTHCARE QUERY

unsupported_query = "What are the symptoms and treatment options for a broken ankle?"
unsupported_result = diabetes_assistant(unsupported_query)
display(pd.DataFrame([unsupported_result]))

In [ ]:
# 12. TEST LOW-CONFIDENCE QUERY

low_confidence_query = "diabetes xyz qwerty unrelated words"
low_confidence_result = diabetes_assistant(low_confidence_query)
display(pd.DataFrame([low_confidence_result]))

In [ ]:
# 13. TEST EMERGENCY-RELATED QUERY

emergency_query = "A person with diabetes is unconscious and not responding. What should I do?"
emergency_result = diabetes_assistant(emergency_query)
display(pd.DataFrame([emergency_result]))

In [ ]:
# 14. INTERACTIVE MULTI-QUERY ASSISTANT
# Run this cell in Google Colab. Enter multiple questions; type 'exit' to stop.

def run_interactive_assistant():
    print("DIABETES CARE INFORMATION ASSISTANT")
    print("Ask a diabetes-related question. Type 'exit' to end the session.")

    conversation_history = []

    while True:
        user_query = input("\nYou: ").strip()

        if user_query.lower() in {"exit", "quit", "stop"}:
            print("Assistant: Session ended. Take care.")
            break

        if not user_query:
            print("Assistant: Please enter a question.")
            continue

        result = diabetes_assistant(user_query, verbose=False)
        print("\nAssistant:", result["response"])
        print(
            f"[Intent: {result['predicted_intent']} | "
            f"Confidence: {result['confidence']} | "
            f"Similarity: {result['similarity']}]"
        )
        conversation_history.append(result)

    if conversation_history:
        print("\nConversation summary:")
        return pd.DataFrame(conversation_history)

    return pd.DataFrame()

# Uncomment the next line to start the interactive session in Colab:
# conversation_df = run_interactive_assistant()

In [ ]:
# 15. OPTIONAL: SIMPLE COLAB TEXT INTERFACE WITH MULTIPLE QUESTIONS
# Run this cell to display a reusable text input interface in Google Colab.

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    query_box = widgets.Text(
        value="",
        placeholder="Enter a diabetes-related question",
        description="Question:",
        layout=widgets.Layout(width="85%")
    )
    ask_button = widgets.Button(
        description="Ask Assistant",
        button_style="success"
    )
    output_area = widgets.Output()
    chat_history = []

    def on_ask_clicked(button):
        query = query_box.value.strip()
        with output_area:
            if not query:
                print("Please enter a question.")
                return

            result = diabetes_assistant(query, verbose=False)
            chat_history.append(result)

            print("You:", query)
            print("Assistant:", result["response"])
            print(
                f"Intent: {result['predicted_intent']} | "
                f"Confidence: {result['confidence']} | "
                f"Similarity: {result['similarity']}"
            )
            print("-" * 80)

            query_box.value = ""

    ask_button.on_click(on_ask_clicked)
    display(widgets.VBox([query_box, ask_button, output_area]))

except ImportError:
    print("ipywidgets is unavailable. Use run_interactive_assistant() instead.")

In [ ]:
# 16. KNOWLEDGE BASE AND CONVERSATION HISTORY EXPORT

knowledge_base_table = pd.DataFrame([
    {"Intent": intent, "Information": information}
    for intent, information in knowledge_base.items()
])
display(knowledge_base_table)

# If the interactive widget has been used, view all submitted queries:
if "chat_history" in globals() and chat_history:
    conversation_history_df = pd.DataFrame(chat_history)
    display(conversation_history_df)
else:
    print("No interactive conversation history to display yet.")

In [ ]:
# 17. REFERENCES
# Centers for Disease Control and Prevention (CDC): https://www.cdc.gov/diabetes/
# National Institute of Diabetes and Digestive and Kidney Diseases: https://www.niddk.nih.gov/health-information/diabetes
# World Health Organization - Diabetes: https://www.who.int/health-topics/diabetes
# NLTK documentation: https://www.nltk.org/
# Scikit-learn documentation: https://scikit-learn.org/stable/

print("Diabetes Care Information Assistant notebook execution completed.")
print("This prototype provides general information only and is not a medical device.")